# Case TechShop - E-commerce Analytics

Notebook principal do case. Cada questão segue o padrão `[MD explicação] -> [CODE] -> [MD análise]`, exceto Q6 (apenas markdown).

# Setup

Imports centralizados, carga do dataset e configuração de caminhos.

In [1]:
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RAW_PATH = Path('../data/raw/ecommerce_vendas.csv')
INTERIM_DIR = Path('../data/interim')
SQL_DIR = Path('../sql')

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

print('Bibliotecas carregadas!')

Bibliotecas carregadas!


# Bloco 2: Q1 — Diagnóstico de Qualidade

Mapeamento de problemas de qualidade do dataset antes de qualquer tratamento.

Checklist coberto: shape e tipos, nulos por coluna, missing disfarçado, duplicatas, domínios categóricos, resumo estatístico numérico, outliers e inconsistências cruzadas.

In [2]:
df = pd.read_csv(RAW_PATH)

# --- 1. Shape e tipos: verificar se o dataset tem as 11 colunas esperadas e se os tipos estão corretos ---
print("=== Shape ===")
print(df.shape)
print("\n=== Tipos ===")
print(df.dtypes)
print("\n=== Amostra ===")
display(df.head())

# --- 2. Nulos: identificar colunas com ausências que impactam volume, receita e análise geográfica ---
print("\n=== Nulos absolutos e porcentagem ===")
nulos = df.isnull().sum()
percentual_nulos = (nulos / len(df) * 100).round(2)
display(pd.DataFrame({'nulos': nulos, 'percentual_%': percentual_nulos})[nulos > 0])

# --- 3. Missing disfarçado: capturar strings vazias que o pandas não converte automaticamente em NaN ---
print("\n=== Missing disfarçado ===")
colunas_texto = df.select_dtypes(include='object').columns
strings_vazias = [(col, (df[col].fillna('').str.strip() == '').sum())
                  for col in colunas_texto
                  if (df[col].fillna('').str.strip() == '').sum() > 0]
if strings_vazias:
    print('\n'.join(f'  {col}: {n}' for col, n in strings_vazias))
else:
    print('Nenhum')

# --- 4. Duplicatas: confirmar se pedido_id é chave primária confiável antes de qualquer agregação ---
print("\n=== Duplicatas em pedido_id ===")
linhas_duplicadas = df[df.duplicated('pedido_id', keep=False)]
print(f"Linhas duplicadas: {len(linhas_duplicadas)}  |  pedido_ids afetados: {linhas_duplicadas['pedido_id'].nunique()}")
display(linhas_duplicadas.sort_values('pedido_id'))

# --- 5. Domínios: detectar valores fora do padrão em status, categoria e uf que fragmentariam agrupamentos ---
print("\n=== Distribuição: status ===")
display(df['status'].value_counts(dropna=False))

print("\n=== Distribuição: categoria ===")
display(df['categoria'].value_counts(dropna=False))

print("\n=== UFs presentes ===")
UFS_VALIDAS = {
    'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG',
    'PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO'
}
ufs_presentes = set(df['uf'].dropna().unique())
ufs_invalidas = ufs_presentes - UFS_VALIDAS
print(f"Únicas (excluindo nulos): {sorted(ufs_presentes)}")
print(f"Fora do domínio: {ufs_invalidas if ufs_invalidas else 'nenhuma'}")

# --- 6. Estatísticas: obter range e distribuição das colunas numéricas como base para o diagnóstico de outliers ---
print("\n=== Resumo estatístico ===")
display(df[['qtd', 'valor_unit', 'desconto_%', 'avaliacao']].describe())

# --- 7a. Validação de domínio: checar violações de regras de negócio com limites fixos conhecidos ---
# Método: limiar fixo. Aplicável a colunas com domínio bem definido (qtd, desconto_%, avaliacao).
# Não detecta outliers estatísticos em valor_unit, que não tem limite superior de negócio definido.
print("\n=== Validação de domínio (regras de negócio) ===")
print(f"qtd <= 0: {(df['qtd'] <= 0).sum()}")
print(f"qtd não-inteira: {((df['qtd'].dropna() % 1) != 0).sum()}")
print(f"valor_unit <= 0: {(df['valor_unit'] <= 0).sum()}")
print(f"desconto_% < 0: {(df['desconto_%'] < 0).sum()}")
print(f"desconto_% > 100: {(df['desconto_%'] > 100).sum()}")
print(f"`avaliacao` fora de [1,5]: {((df['avaliacao'] < 1) | (df['avaliacao'] > 5)).sum()}")

# --- 7b. Outliers estatísticos em valor_unit: IQR, robusto a distribuições assimétricas ---
q1_valor = df['valor_unit'].quantile(0.25)
q3_valor = df['valor_unit'].quantile(0.75)
iqr_valor = q3_valor - q1_valor
limite_inferior_valor = q1_valor - 1.5 * iqr_valor  # IQR preferível ao z-score: distribuição assimétrica (mediana R$259 << média R$484) viola normalidade
limite_superior_valor = q3_valor + 1.5 * iqr_valor  # dois limites: inferior (Q1 - 1,5×IQR) e superior (Q3 + 1,5×IQR)

registros_acima_limite = df[df['valor_unit'] > limite_superior_valor]

print(f"\n=== Outliers estatísticos: valor_unit (método IQR) ===")
print(f"Q1: R$ {q1_valor:.2f}  |  Q3: R$ {q3_valor:.2f}  |  IQR: R$ {iqr_valor:.2f}")
print(f"Limite inferior (Q1 - 1,5 × IQR): R$ {limite_inferior_valor:.2f}")
print(f"Limite superior (Q3 + 1,5 × IQR): R$ {limite_superior_valor:.2f}")
print(f"Registros abaixo do limite inferior: {(df['valor_unit'] < limite_inferior_valor).sum()}")
print(f"  Limite inferior é negativo (R$ {limite_inferior_valor:.2f}); nenhum preço pode assumir valor abaixo de zero.")
print(f"Registros acima do limite superior: {len(registros_acima_limite)}")
if not registros_acima_limite.empty:
    print(f"Produtos envolvidos (acima do limite):")
    display(
        registros_acima_limite
        .groupby(['produto', 'categoria'])['valor_unit']
        .agg(pedidos='count', preco_unitario='first')
        .sort_values('preco_unitario', ascending=False)
        .reset_index()
    )

# --- 7c. Consistência de preço por produto: detectar dispersão de valor_unit dentro do mesmo produto ---
# Método: nunique por produto. Produto com >1 valor_unit distinto indica erro de cadastro/digitação
# (padrão típico: vírgula decimal deslocada, valor 10x). Complementa o IQR, que só pega outlier global.
print("\n=== Dispersão de valor_unit por produto ===")
dispersao_valor_unit = (
    df.dropna(subset=['produto', 'valor_unit'])
      .groupby('produto')['valor_unit']
      .agg(precos_distintos='nunique', valor_min='min', valor_max='max', registros='count')
)
produtos_divergentes = dispersao_valor_unit[dispersao_valor_unit['precos_distintos'] > 1].copy()
print(f"Produtos com mais de um valor_unit: {len(produtos_divergentes)}")
if not produtos_divergentes.empty:
    produtos_divergentes['razao_max_min'] = (produtos_divergentes['valor_max'] / produtos_divergentes['valor_min']).round(2)
    display(produtos_divergentes.sort_values('razao_max_min', ascending=False))

# --- 8. Cruzamento avaliacao x status: verificar se apenas pedidos entregues possuem avaliação preenchida ---
print("\n=== Avaliação por status ===")
avaliacao_por_status = df.groupby('status', observed=True).agg(
    total_pedidos=('pedido_id', 'count'),
    com_avaliacao=('avaliacao', 'count')
)
avaliacao_por_status['pct_avaliados'] = (
    avaliacao_por_status['com_avaliacao'] / avaliacao_por_status['total_pedidos'] * 100
).round(1)
display(avaliacao_por_status)

# --- 9. Datas: detectar formatos inconsistentes que o pandas converterá em NaT e excluirá da análise temporal ---
print("\n=== data_pedido ===")
df['data_pedido_parsed'] = pd.to_datetime(df['data_pedido'], errors='coerce')
datas_invalidas = df['data_pedido_parsed'].isna().sum()

print(f"  Válidas: {len(df) - datas_invalidas}  |  NaT: {datas_invalidas}  |  Range: {df['data_pedido_parsed'].min().date()} a {df['data_pedido_parsed'].max().date()}")
print(f"\nRegistros com formato inválido (dd/mm/yyyy) — {datas_invalidas} no total:")
display(
    df.loc[df['data_pedido_parsed'].isna(), ['pedido_id', 'data_pedido']]
    .sort_values('pedido_id')
    .reset_index(drop=True)
)

df.drop(columns=['data_pedido_parsed'], inplace=True)

=== Shape ===
(1205, 11)

=== Tipos ===
pedido_id        int64
data_pedido        str
cliente_id         str
uf                 str
produto            str
categoria          str
qtd            float64
valor_unit     float64
desconto_%     float64
status             str
avaliacao      float64
dtype: object

=== Amostra ===


,pedido_id,data_pedido,cliente_id,uf,produto,categoria,qtd,valor_unit,desconto_%,status,avaliacao
0,1102,2024-03-02,C191,MG,HD Externo 2TB,Armazenamento,2.0,349.0,0.0,entregue,4.0
1,1947,2024-04-14,C036,PA,Toner HP,Impressoras,1.0,129.0,0.0,entregue,5.0
2,1307,2024-12-07,C031,MG,Mouse Gamer,Periféricos,1.0,89.0,10.0,em_transito,NaN
3,1110,2024-08-07,C079,MG,Headset 7.1,Periféricos,1.0,349.0,20.0,entregue,5.0
4,2062,2024-02-15,C119,SP,Suporte Notebook,Acessórios,2.0,89.9,0.0,entregue,4.0



=== Nulos absolutos e porcentagem ===


,nulos,percentual_%
cliente_id,25,2.07
uf,15,1.24
qtd,12,1.00
desconto_%,30,2.49
avaliacao,241,20.00



=== Missing disfarçado ===
  cliente_id: 25
  uf: 15

=== Duplicatas em pedido_id ===
Linhas duplicadas: 10  |  pedido_ids afetados: 5


...

,precos_distintos,valor_min,valor_max,registros,razao_max_min
produto,,,,,
HD Externo 2TB,2,349.0,3490.0,59,10.0
Headset 7.1,2,349.0,3490.0,68,10.0
"Monitor 24""",2,899.0,8990.0,73,10.0
Mousepad XL,2,59.9,599.0,60,10.0
SSD 500GB,2,189.0,1890.0,52,10.0
Suporte Notebook,2,89.9,899.0,63,10.0



=== Avaliação por status ===


,total_pedidos,com_avaliacao,pct_avaliados
status,,,
Devolvido,2,2,100.0
Entregue,8,8,100.0
cancelado,138,0,0.0
devolvido,82,82,100.0
em_transito,103,0,0.0
entregue,872,872,100.0



=== data_pedido ===
  Válidas: 1185  |  NaT: 20  |  Range: 2024-01-01 a 2024-12-31

Registros com formato inválido (dd/mm/yyyy) — 20 no total:


,pedido_id,data_pedido
0,1022,06/06/2024
1,1054,08/12/2024
2,1107,01/09/2024
3,1115,31/01/2024
4,1197,24/02/2024
5,1208,29/03/2024
6,1209,09/07/2024
7,1478,29/01/2024
8,1604,20/09/2024
9,1696,18/11/2024


### Achados - Q1

#### Achado principal

O dataset tem `1.205` linhas e `11` colunas, e utilizável, mas contém problemas de qualidade que distorcem análises de receita, volume, satisfação e cobertura geográfica.

---

#### Conciliação `[CODE]` vs `[MD analise]`

| Afirmação no MD | Evidência no output do `[CODE]` | Status |
|---|---|---|
| `avaliacao` com `241` ausências (`20,00%`) | tabela de nulos (`241`, `20.00`) | explícito |
| Ausências de `avaliacao` em `cancelado` (`138`) e `em_transito` (`103`) | distribuição de `status` + `Avaliação por status` | explícito |
| Média `3,92/5`, desvio `1,16`, base `964` | `describe`: `3.920124`, `1.155981`, `964` | derivado direto |
| `6` produtos com razão `10x` em `valor_unit` | bloco 7c: `Produtos com mais de um valor_unit: 6` + `razao_max_min = 10.0` | explícito |
| "`3` registros adicionais abaixo do limite estatístico" | não há contagem explícita no output para esses casos | não visível |
| "Total `8` registros com `valor_unit` anômalo" | não há total explícito no output consolidando todos os casos | não visível |

**O que é relevante mencionar para o contexto de negócio**

- Fragmentação de `status`: `Entregue` (`8`) e `Devolvido` (`2`) coexistem com versões em lowercase, isso altera KPIs por agrupamento textual.
- Outlier estatístico isolado não basta: há `118` registros acima do limite IQR, mas `113` pertencem a produtos de ticket alto (Monitor 32" 4K: `57`, Monitor 27" 144Hz: `56`), sobrando `5` suspeitos acima do limite para correção de cadastro.
- Datas inválidas (`20`) viram `NaT`, isso remove linhas de análises temporais se não houver tratamento prévio.

---

#### Evidências-chave

**Campos em branco**

| Coluna | Ausências | % | Impacto downstream |
|---|---|---|---|
| `avaliacao` | `241` | `20,00%` | Ausências concentradas em `cancelado` (`138`) e `em_transito` (`103`), análises de satisfação exigem filtro por `status` antes de calcular médias |
| `desconto_%` | `30` | `2,49%` | Pedidos sem desconto cadastrado ficam fora de qualquer cálculo de margem líquida |
| `cliente_id` | `25` | `2,07%` | Pedidos anônimos não entram em segmentação nem análise comportamental por cliente |
| `uf` | `15` | `1,24%` | Pedidos sem UF ficam fora de relatórios regionais |
| `qtd` | `12` | `1,00%` | Linhas sem quantidade não contribuem para volume nem receita |

Strings vazias confirmaram as mesmas `25` e `15` ausências, sem ocultas adicionais. `desconto_%` não apresentou violações de domínio e o máximo observado foi `20%`. Os `964` pedidos com avaliação têm média `3,92/5` e desvio padrão `1,16`.

**Pedidos duplicados**

`10` linhas correspondem a `5` pedidos distintos, contagens e somatórios de receita ficam inflados. Dois casos críticos: pedido `1113` com `cliente_id` divergente entre cópias e pedido `1885` com duplicata junto de inconsistência de categoria (`"Monitores"` vs `"monitores"`).

**Grafias inconsistentes**

`status` tem `10` registros com inicial maiúscula (`Entregue`: `8`, `Devolvido`: `2`), isso fragmenta agrupamentos. `categoria` também tem variantes em minúsculas e duplica grupos sem erro de execução.

**Quantidades inválidas e datas mal formadas**

Há `5` registros com `qtd <= 0` (mínimo `-1`). Em datas, `20` registros estão em `dd/mm/yyyy` e viram `NaT` no parse padrão. Faixa válida observada: `2024-01-01` a `2024-12-31`.

**Erros de preço unitário por produto**

No diagnóstico IQR, o limite superior de `valor_unit` é `R$ 1.304,00` e há `118` registros acima desse limite. Desse total, `113` são de produtos de ticket alto (Monitor 32" 4K `57` + Monitor 27" 144Hz `56`) e `5` ficam concentrados em padrões típicos de cadastro `10x` acima do limite (Monitor 24", HD Externo 2TB, Headset 7.1, SSD 500GB).

No diagnóstico intra-produto (bloco 7c), há `6` produtos com dois `valor_unit` e razão `10x`: Monitor 24", HD Externo 2TB, Headset 7.1, SSD 500GB, Mousepad XL e Suporte Notebook. Para os dois últimos, o output do bloco 7c confirma a divergência `10x`, mas não expõe contagem explícita de linhas infladas no mesmo quadro.

---

#### O que isso significa para o negócio

1. Receita pode ser superestimada, porque `5` pedidos distintos estão duplicados no nível de linha.
2. Relatórios regionais perdem cobertura, porque `15` pedidos sem `uf` não entram em análises por estado.
3. Indicadores de satisfação ficam enviesados se não houver filtro por `status`, pois há `241` ausências concentradas em pedidos não entregues.
4. Agrupamentos por texto podem quebrar silenciosamente por variação de capitalização em `status` e `categoria`.

---

#### Ressalvas

- O output confirma `13` UFs presentes na base e nenhuma UF fora do domínio esperado.
- O diagnóstico descreve o dataset como exportado, não permite inferir com certeza a origem operacional de cada problema.

---

#### Próximo passo, Q2

Os achados seguem catalogados em `memory-bank/data-issues.md` (DI-001 a DI-012). Ordem de tratamento recomendada:

1. **DI-002**: remover duplicatas de `pedido_id`, para `1113`, preservar a cópia com `cliente_id` preenchido.
2. **DI-012**: corrigir inconsistências de `valor_unit` com padrão `10x`, priorizando casos explicitamente suspeitos no IQR.
3. **DI-003**: excluir ou corrigir `qtd <= 0`.
4. **DI-008 / DI-009**: normalizar capitalização de `status` e `categoria`.
5. **DI-010**: reprocessar `data_pedido` suportando ambos os formatos.
6. **DI-001 / DI-006 / DI-007**: definir estratégia para campos com ausência (exclusão, marcação ou imputação).

# Bloco 3: Q2 — Tratamento e Justificativa

Q2 trata os `12` problemas catalogados em DI-001–DI-012 a partir do dataset bruto `data/raw/ecommerce_vendas.csv` (imutável). O dataset tratado é salvo em `data/interim/ecommerce_tratado.csv`.

**Ordem de tratamento** (H → M → L):

1. **DI-002** (H) — Duplicatas de `pedido_id`: deduplicar preservando a cópia com `cliente_id` preenchido.
2. **DI-012** (H) — `valor_unit` com padrão `10x`: corrigir dividindo por `10` para `6` produtos identificados.
3. **DI-003** (H) — `qtd <= 0`: remover `5` registros inválidos.
4. **DI-008** (M) — `status` com inicial maiúscula: normalizar para minúsculas (`str.lower()`).
5. **DI-009** (M) — `categoria` com variantes em minúsculas: normalizar para title case (`str.title()`).
6. **DI-010** (M) — `data_pedido` em `dd/mm/yyyy`: reprocessar suportando ambos os formatos.
7. **DI-007** (M) — `qtd` nulos: remover `12` linhas; converter `qtd` para `Int64` (**DI-011**).
8. **DI-001** (H) — `cliente_id` nulos: manter com flag `cliente_anonimo`.
9. **DI-004** (M) — `desconto_%` nulos: imputar `0` (sem desconto aplicado, assumido).
10. **DI-005** (M) — `avaliacao` nulos: manter como `NaN` (padrão estrutural — `cancelado`/`em_transito`).
11. **DI-006** (M) — `uf` nulos: manter como `NaN` (sem dado para imputar).
12. Renomear `desconto_%` → `desconto_pct`; derivar coluna `receita`.

In [3]:
# ============================================================
# Q2 - Tratamento de qualidade
# ============================================================
print(f"Shape inicial: {df.shape}")

# --- DI-002: Remover duplicatas de pedido_id ---
# Ordenar para que cliente_id preenchido venha antes de NaN (pedido 1113)
linhas_antes_dedup = len(df)
df = (df
      .sort_values(['pedido_id', 'cliente_id'], na_position='last')
      .drop_duplicates(subset='pedido_id', keep='first')
      .reset_index(drop=True))
print(f"\nDI-002 | Duplicatas removidas: {linhas_antes_dedup - len(df)} | Shape: {df.shape}")
print(f"  Pedido 1113 cliente_id após dedup: {df.loc[df['pedido_id'] == 1113, 'cliente_id'].values}")

# --- DI-012: Corrigir valor_unit com padrão 10x ---
# Para cada um dos 6 produtos, o menor valor_unit é o correto; os inflados têm razão exata 10x.
produtos_10x = ['HD Externo 2TB', 'Headset 7.1', 'Monitor 24"', 'Mousepad XL', 'SSD 500GB', 'Suporte Notebook']
precos_corrigidos = 0
print(f"\nDI-012 | Correção de valor_unit 10x:")
for produto in produtos_10x:
    preco_min = df.loc[df['produto'] == produto, 'valor_unit'].min()
    preco_inflado = round(preco_min * 10, 2)
    pedidos_preco_inflado = (df['produto'] == produto) & (df['valor_unit'].round(2) == preco_inflado)
    precos_afetados = pedidos_preco_inflado.sum()
    df.loc[pedidos_preco_inflado, 'valor_unit'] = preco_min
    precos_corrigidos += precos_afetados
    print(f"  {produto}: {precos_afetados} registro(s) de R${preco_inflado:.2f} → R${preco_min:.2f}")
print(f"DI-012 | Total corrigidos: {precos_corrigidos}")
print(f"  valor_unit range após correção: R${df['valor_unit'].min():.2f} a R${df['valor_unit'].max():.2f}")

# --- DI-003: Remover registros com qtd <= 0 ---
linhas_antes_qtd_invalida = len(df)
pedidos_qtd_invalida = df['qtd'] <= 0
print(f"\nDI-003 | Registros com qtd <= 0: {pedidos_qtd_invalida.sum()}")
df = df[~pedidos_qtd_invalida].reset_index(drop=True)
print(f"DI-003 | Removidos: {linhas_antes_qtd_invalida - len(df)} | Shape: {df.shape}")

# --- DI-008: Normalizar status para minúsculas ---
registros_status_maiusculo = (df['status'] != df['status'].str.lower()).sum()
df['status'] = df['status'].str.lower()
print(f"\nDI-008 | Registros normalizados em status: {registros_status_maiusculo}")
display(df['status'].value_counts())

# --- DI-009: Normalizar categoria para title case ---
registros_categoria_minuscula = (df['categoria'] != df['categoria'].str.title()).sum()
df['categoria'] = df['categoria'].str.title()
print(f"\nDI-009 | Registros normalizados em categoria: {registros_categoria_minuscula}")
display(df['categoria'].value_counts())

# --- DI-010: Reprocessar data_pedido suportando yyyy-mm-dd e dd/mm/yyyy ---
# Regex converte dd/mm/yyyy → yyyy-mm-dd antes do parse único
df['data_pedido'] = pd.to_datetime(
    df['data_pedido'].str.replace(
        r'^(\d{2})/(\d{2})/(\d{4})$', r'\3-\2-\1', regex=True
    ),
    format='%Y-%m-%d'
)
datas_invalidas_restantes = df['data_pedido'].isna().sum()
print(f"\nDI-010 | NaT após reprocessamento: {datas_invalidas_restantes}")
print(f"  Range: {df['data_pedido'].min().date()} a {df['data_pedido'].max().date()}")

# --- DI-007 + DI-011: Remover qtd nulos e converter para Int64 ---
linhas_antes_qtd_nulo = len(df)
df = df.dropna(subset=['qtd']).reset_index(drop=True)
df['qtd'] = df['qtd'].astype('Int64')
print(f"\nDI-007 | qtd nulos removidos: {linhas_antes_qtd_nulo - len(df)} | Shape: {df.shape}")
print(f"DI-011 | qtd dtype: {df['qtd'].dtype}")

# --- DI-001: Manter cliente_id nulos com flag ---
df['cliente_anonimo'] = df['cliente_id'].isna()
print(f"\nDI-001 | Pedidos anônimos (cliente_id nulo): {df['cliente_anonimo'].sum()} — mantidos com flag cliente_anonimo")

# --- DI-004: Imputar desconto_% nulos com 0 ---
desconto_nulos = df['desconto_%'].isna().sum()
df['desconto_%'] = df['desconto_%'].fillna(0.0)
print(f"\nDI-004 | desconto_% imputados com 0: {desconto_nulos}")

# --- DI-005 / DI-006: Manter ausências estruturais ---
print(f"\nDI-005 | avaliacao nulos mantidos: {df['avaliacao'].isna().sum()} (estrutural: cancelado/em_transito)")
print(f"DI-006 | uf nulos mantidos: {df['uf'].isna().sum()} (sem dado para imputar — excluídos de análises geográficas)")

# --- Renomear desconto_% → desconto_pct (nome canônico downstream) ---
df = df.rename(columns={'desconto_%': 'desconto_pct'})

# --- Derivar coluna receita ---
df['receita'] = (df['qtd'] * df['valor_unit'] * (1 - df['desconto_pct'] / 100)).round(2)
print(f"\nReceita derivada — range: R${df['receita'].min():.2f} a R${df['receita'].max():.2f}")

# --- Resumo final ---
print(f"\n=== Shape final: {df.shape} ===")
print(f"Diff: 1205×11 → {df.shape[0]}×{df.shape[1]}")
print("\n=== Tipos finais ===")
print(df.dtypes)
print("\n=== Nulos restantes ===")
nulos_restantes = df.isnull().sum()
display(pd.DataFrame({'nulos': nulos_restantes[nulos_restantes > 0]}))

# --- Salvar dataset tratado ---
INTERIM_PATH = INTERIM_DIR / 'ecommerce_tratado.csv'
df.to_csv(INTERIM_PATH, index=False, encoding='utf-8')
print(f"\nDataset tratado salvo em: {INTERIM_PATH}")

Shape inicial: (1205, 11)

DI-002 | Duplicatas removidas: 5 | Shape: (1200, 11)
  Pedido 1113 cliente_id após dedup: <StringArray>
['C019']
Length: 1, dtype: str

DI-012 | Correção de valor_unit 10x:
  HD Externo 2TB: 1 registro(s) de R$3490.00 → R$349.00
  Headset 7.1: 2 registro(s) de R$3490.00 → R$349.00
  Monitor 24": 1 registro(s) de R$8990.00 → R$899.00
  Mousepad XL: 1 registro(s) de R$599.00 → R$59.90
  SSD 500GB: 1 registro(s) de R$1890.00 → R$189.00
  Suporte Notebook: 2 registro(s) de R$899.00 → R$89.90
DI-012 | Total corrigidos: 8
  valor_unit range após correção: R$39.90 a R$2499.00

DI-003 | Registros com qtd <= 0: 5
DI-003 | Removidos: 5 | Shape: (1195, 11)

DI-008 | Registros normalizados em status: 10


status
entregue       873
cancelado      137
em_transito    101
devolvido       84
Name: count, dtype: int64


DI-009 | Registros normalizados em categoria: 14


categoria
Periféricos      270
Acessórios       231
Armazenamento    219
Monitores        183
Impressoras      171
Câmeras          121
Name: count, dtype: int64


DI-010 | NaT após reprocessamento: 0
  Range: 2024-01-01 a 2024-12-31

DI-007 | qtd nulos removidos: 12 | Shape: (1183, 11)
DI-011 | qtd dtype: Int64

DI-001 | Pedidos anônimos (cliente_id nulo): 24 — mantidos com flag cliente_anonimo

DI-004 | desconto_% imputados com 0: 28

DI-005 | avaliacao nulos mantidos: 236 (estrutural: cancelado/em_transito)
DI-006 | uf nulos mantidos: 15 (sem dado para imputar — excluídos de análises geográficas)

Receita derivada — range: R$31.92 a R$11870.25

=== Shape final: (1183, 13) ===
Diff: 1205×11 → 1183×13

=== Tipos finais ===
pedido_id                   int64
data_pedido        datetime64[us]
cliente_id                    str
uf                            str
produto                       str
categoria                     str
qtd                         Int64
valor_unit                float64
desconto_pct              float64
status                        str
avaliacao                 float64
cliente_anonimo              bool
receita              

,nulos
cliente_id,24
uf,15
avaliacao,236



Dataset tratado salvo em: ..\data\interim\ecommerce_tratado.csv


# Bloco 4: Q3 - SQL

## Q3 - Consultas SQL

_A preencher via `/consultar-sql`._

# Bloco 5: Q4 - Negócio

## Q4 - Analise de Negocio Aberta

_A preencher via `/analisar-negocio`._

# Bloco 6: Q5 - Debug

## Q5 - Encontre o Erro

_A preencher via `/encontrar-erro`._

# Bloco 7: Q6 - Modelagem Dimensional

## Q6 - Modelagem

_A preencher via `/modelar-dimensional` (markdown-only)._

# Bloco 8: Q7 - Insight

## Q7 - Pergunta Livre

_A preencher via `/insight-livre`._